# Lumis 10: Photo check (no skin / nothing detected)

The model always picks one of its five conditions, even for a photo of a desk (a random-noise test image came back
as "Most likely Eczema"). Two checks now run **before** a result is shown:

| Check | Measure | App message |
|---|---|---|
| 1. Is there skin? | Share of skin-colored pixels (the notebook 06 rule) | **No skin detected.** |
| 2. Does it look like any condition Lumis was trained on? | Energy score of the model output (Liu et al., 2020): high energy = unlike the training photos | **Nothing detected.** (and: this does not mean your skin is healthy) |

**How the thresholds are set:** on the **validation** photos (real condition photos), each check must let through
`PASS_TARGET` of them (default 99%), so real conditions are almost never turned away. Then we measure how many
non-skin and healthy-skin photos each check catches.

**Photos used to measure it:**
- validation photos (sets the thresholds),
- test photos split by skin type, to check that darker skin is not turned away more often (only pass rates are measured; nothing about the model is changed),
- the 165 non-skin images removed from DermNet in notebook 06 (microscopy, insects),
- optional: your own phone photos in `Drive/Lumis/data/photo_check/` subfolders `healthy_skin/`, `not_skin/` and `real_conditions/` (phone photos of one of the five conditions, which should pass). JPEG or PNG.

CPU runtime is fine. About 5 minutes.

In [ ]:
PASS_TARGET = 0.99                 # share of real condition photos each check must let through
TARGETS = [0.95, 0.975, 0.99, 0.995]
MY_PHOTOS = 'data/photo_check'     # inside Drive/Lumis; optional subfolders healthy_skin/, not_skin/, real_conditions/

In [ ]:
!pip install -q onnxruntime simplejpeg
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'

In [ ]:
import os, json, zipfile, glob, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

LOCAL = '/content/dermnet'
MODEL_DIR = f'{BASE}/exports/model'
if not os.path.exists(LOCAL):
    with zipfile.ZipFile(f'{BASE}/data/dermnet.zip') as z:
        z.extractall(LOCAL)

cfg = json.load(open(f'{MODEL_DIR}/lumis_model_config.json'))
RUN, SIZE = cfg['run'], cfg['img_size']
split = pd.read_csv(f"{BASE}/data/lumis_split_{cfg['split']}.csv")
val = split[split.split == 'val'].reset_index(drop=True)
test = split[split.split == 'test'].reset_index(drop=True)
test = test.merge(pd.read_csv(f'{BASE}/data/skin_tone_labels_round1.csv')[['path', 'tone']], on='path', how='left')
val_logits = np.load(f'{BASE}/exports/runs/{RUN}/val_logits.npy')
test_logits = np.load(f'{BASE}/exports/final_test/test_logits.npy')
assert len(val_logits) == len(val) and len(test_logits) == len(test)
removed = pd.read_csv(f'{BASE}/exports/removed_nonskin_v2.csv')

def my_photos(sub):
    folder = f'{BASE}/{MY_PHOTOS}/{sub}'
    return sorted(p for p in glob.glob(f'{folder}/*') if os.path.isfile(p))

mine = {k: my_photos(k) for k in ['healthy_skin', 'not_skin', 'real_conditions']}
print(f'{len(val)} validation, {len(test)} test, {len(removed)} DermNet non-skin images')
print(f"Your photos: {len(mine['healthy_skin'])} healthy skin, {len(mine['not_skin'])} not skin, {len(mine['real_conditions'])} real conditions")

The next cell is `backend/app/ml/predictor.py`, unchanged, so the notebook measures exactly what the server will run.

In [ ]:
%%writefile predictor.py
"""Lumis skin-condition predictor.

Runs the exported EfficientNet-B0 (ONNX) and turns its output into what the app shows:
a condition, a calibrated confidence and a band (high / moderate / refer).

Preprocessing reproduces training exactly (ml/03_train.ipynb), checked pixel for pixel in ml/09_export.ipynb:
  1. Decode to RGB. JPEG uses libjpeg-turbo with the fast integer DCT, which is what
     tf.io.decode_image uses by default. PNG and WebP are decoded by Pillow.
  2. Resize to 224 x 224 like tf.image.resize: bilinear, half-pixel centers, no antialiasing,
     aspect ratio NOT kept (training stretched every image to a square).
  3. Cast to uint8 (truncation), then to float32 in the 0-255 range. EfficientNet scales and
     normalizes inside the model, so nothing else is done here.
One deliberate difference from training: phone photos are turned upright using their EXIF
orientation tag, because phones often save the pixels sideways.

Photo check (notebook 10), before any result is shown:
  no_skin           too few skin-colored pixels (the notebook 06 rule: YCbCr Cr 130-190, Cb 70-130)
  nothing_detected  the photo is unlike any of the training photos (energy score above the threshold)
Thresholds live in models/photo_check.json. Without that file every photo passes.

Output steps, decided in notebooks 04, 04b and 08:
  adjusted logits = logits + logit_adjust     (prior adjustment, training mix, tau 0.75)
  probabilities   = softmax(adjusted logits / temperature)
  band            = high if confidence >= high_from, moderate if >= refer_below, else refer
  scabies (any class in cap_at_moderate) is never shown as high: at most moderate.
"""
from __future__ import annotations

import hashlib
import io
import json
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import numpy as np
import simplejpeg
from PIL import Image, UnidentifiedImageError

MAX_PIXELS = 50_000_000          # about 8,000 x 6,000; larger uploads are refused
SUPPORTED_FORMATS = {"JPEG", "PNG", "WEBP"}


class UnsupportedImageError(ValueError):
    """The upload is not an image Lumis can read."""


# ---------------------------------------------------------------- preprocessing

def _apply_exif_orientation(rgb: np.ndarray, orientation: int) -> np.ndarray:
    """Same result as PIL.ImageOps.exif_transpose, on a numpy array."""
    if orientation == 2:
        return rgb[:, ::-1]
    if orientation == 3:
        return rgb[::-1, ::-1]
    if orientation == 4:
        return rgb[::-1]
    if orientation == 5:
        return rgb.transpose(1, 0, 2)
    if orientation == 6:
        return np.rot90(rgb, -1)
    if orientation == 7:
        return rgb.transpose(1, 0, 2)[::-1, ::-1]
    if orientation == 8:
        return np.rot90(rgb, 1)
    return rgb


def decode_image(data: bytes, apply_exif: bool = True) -> np.ndarray:
    """Bytes of a JPEG, PNG or WebP file -> uint8 array of shape (height, width, 3)."""
    try:
        im = Image.open(io.BytesIO(data))
        fmt = im.format
        if fmt not in SUPPORTED_FORMATS:
            raise UnsupportedImageError(f"Unsupported image format: {fmt}. Use JPEG, PNG or WebP.")
        if im.width * im.height > MAX_PIXELS:
            raise UnsupportedImageError("Image is too large.")
        orientation = im.getexif().get(0x0112, 1) if apply_exif else 1
        rgb = None
        if fmt == "JPEG":
            try:
                rgb = simplejpeg.decode_jpeg(data, colorspace="RGB", fastdct=True, fastupsample=False)
            except ValueError:
                rgb = None       # unusual JPEGs (for example CMYK): fall back to Pillow below
        if rgb is None:
            rgb = np.asarray(im.convert("RGB"))
    except UnsupportedImageError:
        raise
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError) as exc:
        raise UnsupportedImageError("The file could not be read as an image.") from exc
    return _apply_exif_orientation(rgb, orientation)


def resize_like_tf(img: np.ndarray, size: int) -> np.ndarray:
    """tf.image.resize(img, (size, size)) with its defaults: bilinear, half-pixel centers,
    no antialiasing. Returns float32."""
    h, w = img.shape[:2]

    def coords(n_in: int, n_out: int):
        src = (np.arange(n_out, dtype=np.float32) + 0.5) * np.float32(n_in / n_out) - 0.5
        lo = np.floor(src)
        frac = (src - lo).astype(np.float32)
        lo_i = np.clip(lo, 0, n_in - 1).astype(np.int64)
        hi_i = np.clip(lo + 1, 0, n_in - 1).astype(np.int64)
        return lo_i, hi_i, frac

    y0, y1, fy = coords(h, size)
    x0, x1, fx = coords(w, size)
    img = img.astype(np.float32)
    fx3, fy3 = fx[None, :, None], fy[:, None, None]
    top = img[y0][:, x0] * (1 - fx3) + img[y0][:, x1] * fx3
    bottom = img[y1][:, x0] * (1 - fx3) + img[y1][:, x1] * fx3
    return top * (1 - fy3) + bottom * fy3


def to_model_input(rgb: np.ndarray, size: int = 224) -> np.ndarray:
    """Decoded RGB array -> float32 model input of shape (1, size, size, 3), values 0-255."""
    resized = resize_like_tf(rgb, size).astype(np.uint8)       # truncation, as in training
    return resized.astype(np.float32)[None]


def preprocess(data: bytes, size: int = 224, apply_exif: bool = True) -> np.ndarray:
    """Image file bytes -> float32 model input of shape (1, size, size, 3), values 0-255."""
    return to_model_input(decode_image(data, apply_exif=apply_exif), size)


# ---------------------------------------------------------------- photo check

def skin_fraction(rgb: np.ndarray) -> float:
    """Share of skin-colored pixels, the rule from notebook 06: on a 128 x 128 copy, YCbCr chroma
    Cr 130-190 and Cb 70-130. Deliberately wide so red rashes and darker skin count as skin."""
    small = Image.fromarray(np.ascontiguousarray(rgb)).resize((128, 128)).convert("YCbCr")
    ycc = np.asarray(small, dtype=np.int16)
    cb, cr = ycc[..., 1], ycc[..., 2]
    return float(((cr >= 130) & (cr <= 190) & (cb >= 70) & (cb <= 130)).mean())


def energy_score(logits: np.ndarray) -> float:
    """Energy of the raw logits, -logsumexp (Liu et al., 2020). Photos like the training photos get low
    (more negative) energy; photos unlike anything the model was trained on get higher energy."""
    z = np.asarray(logits, dtype=np.float64).reshape(-1)
    m = z.max()
    return float(-(m + np.log(np.exp(z - m).sum())))


@dataclass(frozen=True)
class PhotoCheckConfig:
    min_skin_fraction: float
    max_energy: float

    @classmethod
    def load(cls, path: str | Path) -> "PhotoCheckConfig":
        c = json.loads(Path(path).read_text())
        return cls(min_skin_fraction=float(c["min_skin_fraction"]), max_energy=float(c["max_energy"]))

    def status(self, skin: float, energy: float) -> str:
        if skin < self.min_skin_fraction:
            return "no_skin"
        if energy > self.max_energy:
            return "nothing_detected"
        return "ok"


# ---------------------------------------------------------------- model output -> result

@dataclass(frozen=True)
class ModelConfig:
    model_version: str
    classes: list[str]
    img_size: int
    temperature: float
    logit_adjust: list[float]
    high_from: float
    refer_below: float
    cap_at_moderate: list[str] = field(default_factory=list)
    onnx_sha256: str | None = None
    model_file: str = "model.onnx"

    @classmethod
    def load(cls, path: str | Path) -> "ModelConfig":
        c = json.loads(Path(path).read_text())
        return cls(model_version=c["model_version"], classes=list(c["classes"]), img_size=int(c["img_size"]),
                   temperature=float(c["temperature"]), logit_adjust=[float(v) for v in c["logit_adjust"]],
                   high_from=float(c["bands"]["high_from"]), refer_below=float(c["bands"]["refer_below"]),
                   cap_at_moderate=list(c.get("cap_at_moderate", [])), onnx_sha256=c.get("onnx_sha256"),
                   model_file=c.get("model_file", "model.onnx"))


@dataclass(frozen=True)
class Prediction:
    condition: str              # most likely class (also given for 'refer', but the app shows it as inconclusive)
    confidence: float           # calibrated probability of that class, 0-1
    band: str                   # 'high' | 'moderate' | 'refer'
    probabilities: dict[str, float]
    capped: bool                # True if the scabies rule lowered the band from high to moderate
    model_version: str
    status: str = "ok"          # 'ok' | 'no_skin' | 'nothing_detected' (photo check)
    skin_fraction: float | None = None
    energy: float | None = None

    def to_dict(self) -> dict:
        return asdict(self)


def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def postprocess(logits: np.ndarray, cfg: ModelConfig) -> Prediction:
    """Raw model logits for one image (shape (K,)) -> Prediction."""
    logits = np.asarray(logits, dtype=np.float64).reshape(-1)
    probs = softmax((logits + np.asarray(cfg.logit_adjust)) / cfg.temperature)
    i = int(probs.argmax())
    conf = float(probs[i])
    condition = cfg.classes[i]
    if conf >= cfg.high_from:
        band = "high"
    elif conf >= cfg.refer_below:
        band = "moderate"
    else:
        band = "refer"
    capped = band == "high" and condition in cfg.cap_at_moderate
    if capped:
        band = "moderate"
    return Prediction(condition=condition, confidence=round(conf, 4), band=band,
                      probabilities={c: round(float(p), 4) for c, p in zip(cfg.classes, probs)},
                      capped=capped, model_version=cfg.model_version)


# ---------------------------------------------------------------- the predictor

def sha256_of(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


class LumisPredictor:
    """Load once at startup, then call predict() per upload. Safe to share across requests."""

    def __init__(self, model_path: str | Path, config_path: str | Path, verify_checksum: bool = True,
                 photo_check_path: str | Path | None = None):
        import onnxruntime as ort

        self.config = ModelConfig.load(config_path)
        self.photo_check = PhotoCheckConfig.load(photo_check_path) if photo_check_path else None
        if verify_checksum and self.config.onnx_sha256:
            actual = sha256_of(model_path)
            if actual != self.config.onnx_sha256:
                raise RuntimeError(f"Model file does not match its config (sha256 {actual[:12]}..., "
                                   f"expected {self.config.onnx_sha256[:12]}...). Re-download both from Drive.")
        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name

    @classmethod
    def from_dir(cls, models_dir: str | Path, config_name: str = "lumis_model_config.json") -> "LumisPredictor":
        """Load the config in models_dir and the ONNX file it names."""
        models_dir = Path(models_dir)
        config_path = models_dir / config_name
        photo_check = models_dir / "photo_check.json"
        return cls(models_dir / ModelConfig.load(config_path).model_file, config_path,
                   photo_check_path=photo_check if photo_check.exists() else None)

    def logits(self, batch: np.ndarray) -> np.ndarray:
        return self.session.run(None, {self.input_name: batch.astype(np.float32)})[0]

    def predict(self, data: bytes) -> Prediction:
        rgb = decode_image(data)
        skin = skin_fraction(rgb)
        logits = self.logits(to_model_input(rgb, self.config.img_size))[0]
        energy = energy_score(logits)
        status = self.photo_check.status(skin, energy) if self.photo_check else "ok"
        return replace(postprocess(logits, self.config), status=status,
                       skin_fraction=round(skin, 4), energy=round(energy, 4))

In [ ]:
import importlib, predictor
importlib.reload(predictor)
from predictor import (LumisPredictor, UnsupportedImageError, decode_image, energy_score, skin_fraction,
                       to_model_input)

pred = LumisPredictor.from_dir(MODEL_DIR)

def measure(paths, logits=None):
    """Skin share and energy for each photo. Saved logits are used where they exist (same model)."""
    rows = []
    for i, p in enumerate(paths):
        try:
            rgb = decode_image(open(p, 'rb').read())
        except UnsupportedImageError:
            rows.append({'path': p, 'readable': False, 'skin': np.nan, 'energy': np.nan})
            continue
        lg = logits[i] if logits is not None else pred.logits(to_model_input(rgb, SIZE))[0]
        rows.append({'path': p, 'readable': True, 'skin': skin_fraction(rgb), 'energy': energy_score(lg)})
    return pd.DataFrame(rows)

t0 = time.time()
sets = {
    'Validation: condition photos': measure([os.path.join(LOCAL, p) for p in val.path], val_logits),
    'DermNet non-skin (microscopy, insects)': measure([os.path.join(LOCAL, p) for p in removed.path]),
}
test_m = measure([os.path.join(LOCAL, p) for p in test.path], test_logits).assign(tone=test.tone.values)
for tone in ['I-II', 'III-IV', 'V-VI']:
    sets[f'Test: condition photos, skin type {tone}'] = test_m[test_m.tone == tone]
if mine['healthy_skin']:
    sets['Your photos: healthy skin'] = measure(mine['healthy_skin'])
if mine['not_skin']:
    sets['Your photos: not skin'] = measure(mine['not_skin'])
if mine['real_conditions']:
    sets['Your photos: real conditions (should pass)'] = measure(mine['real_conditions'])
for name, d in sets.items():
    if (~d.readable).any():
        print(f'{name}: {(~d.readable).sum()} files could not be read (HEIC photos must be saved as JPEG)')
print(f'Measured in {time.time() - t0:.0f} s')

## 1. Where the photos fall

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for name, d in sets.items():
    if name.startswith('Test'):
        continue
    d = d[d.readable]
    ax[0].hist(d.skin, bins=30, range=(0, 1), alpha=0.5, density=True, label=f'{name} (n={len(d)})')
    ax[1].hist(d.energy, bins=30, alpha=0.5, density=True, label=name)
ax[0].set_xlabel('share of skin-colored pixels'); ax[0].set_title('Check 1: skin')
ax[1].set_xlabel('energy (higher = less like the training photos)'); ax[1].set_title('Check 2: like a trained condition?')
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/photo_check_distributions.png', dpi=110)
plt.show()

## 2. Thresholds and what they catch

For each target, the skin threshold is the value that `target` of validation photos are above, and the energy
threshold the value that `target` of them are below. "Passes" = gets a normal result.

In [ ]:
def thresholds(p):
    v = sets['Validation: condition photos']
    return float(v.skin.quantile(1 - p)), float(v.energy.quantile(p))

def outcomes(d, skin_t, energy_t):
    d = d[d.readable]
    no_skin = d.skin < skin_t
    nothing = ~no_skin & (d.energy > energy_t)
    return {'n': len(d), 'passes': (~no_skin & ~nothing).mean(), 'no_skin': no_skin.mean(), 'nothing_detected': nothing.mean()}

summary = pd.DataFrame({f'{p:.1%}': {name: outcomes(d, *thresholds(p))['passes'] for name, d in sets.items()}
                        for p in TARGETS})
print('Share of photos that PASS (get a normal result), by target:')
display((summary * 100).round(1))

skin_t, energy_t = thresholds(PASS_TARGET)
chosen = pd.DataFrame({name: outcomes(d, skin_t, energy_t) for name, d in sets.items()}).T
print(f'\nChosen target {PASS_TARGET:.1%}: no skin below {skin_t:.3f} skin share; nothing detected above energy {energy_t:.3f}')
display(chosen.assign(**{c: (chosen[c] * 100).round(1) for c in ['passes', 'no_skin', 'nothing_detected']}))

## 3. Look at the edge cases

Real condition photos that would be turned away, and non-skin photos that would still get a result.

In [ ]:
from PIL import Image

def grid(df, title, n=16):
    df = df.head(n)
    if df.empty:
        print(title, ': none'); return
    cols = 8
    fig, axes = plt.subplots(int(np.ceil(len(df) / cols)), cols, figsize=(20, 2.8 * int(np.ceil(len(df) / cols))), squeeze=False)
    for a in axes.flat:
        a.axis('off')
    for a, (_, r) in zip(axes.flat, df.iterrows()):
        a.imshow(Image.open(r.path).convert('RGB'))
        a.set_title(f"skin {r.skin:.2f}  energy {r.energy:.1f}", fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.show()

v = sets['Validation: condition photos']
grid(v[v.skin < skin_t].sort_values('skin'), 'Condition photos that would get "No skin detected"')
grid(v[(v.skin >= skin_t) & (v.energy > energy_t)].sort_values('energy', ascending=False), 'Condition photos that would get "Nothing detected"')
ns = sets['DermNet non-skin (microscopy, insects)']
grid(ns[(ns.skin >= skin_t) & (ns.energy <= energy_t)], 'Non-skin images that would still get a result')
for name in ['Your photos: healthy skin', 'Your photos: not skin']:
    if name in sets:
        d = sets[name][sets[name].readable]
        grid(d[(d.skin >= skin_t) & (d.energy <= energy_t)], f'{name}: would still get a result')
name = 'Your photos: real conditions (should pass)'
if name in sets:
    d = sets[name][sets[name].readable]
    grid(d[(d.skin < skin_t) | (d.energy > energy_t)], f'{name}: would be turned away')

## 4. Save the thresholds for the backend

In [ ]:
fair = {t: round(float(chosen.loc[f'Test: condition photos, skin type {t}', 'passes']), 4) for t in ['I-II', 'III-IV', 'V-VI']}
out = {
    'min_skin_fraction': round(skin_t, 4),
    'max_energy': round(energy_t, 4),
    'pass_target': PASS_TARGET,
    'model_version': cfg['model_version'],
    'set_on': 'validation condition photos',
    'measured': {name: {k: (round(float(v), 4) if k != 'n' else int(v)) for k, v in row.items()}
                 for name, row in chosen.iterrows()},
    'test_pass_rate_by_skin_type': fair,
    'created_at': time.strftime('%Y-%m-%d %H:%M'),
}
json.dump(out, open(f'{MODEL_DIR}/photo_check.json', 'w'), indent=1)
(summary * 100).round(2).to_csv(f'{MODEL_DIR}/photo_check_targets.csv')
print(json.dumps({k: out[k] for k in ['min_skin_fraction', 'max_energy', 'pass_target', 'test_pass_rate_by_skin_type']}, indent=1))
print('\nSaved photo_check.json: download it into backend/models/')